# Alpamayo 2 Super Expert Inference

Run this notebook from the repository root with the `Alpamayo 2 Super` kernel. Set `ALPAMAYO2_SUPER_MODEL_ID` to a Hugging Face model id or local release checkpoint before starting the kernel. The notebook selects the validated six-camera/four-frame trajectory input profile before model preparation.


In [ ]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

from alpamayo2_super import helper
from alpamayo2_super.common.constants import PUBLIC_MODEL_ID
from alpamayo2_super.inference_smoke import resolve_project_path, validate_model_id
from alpamayo2_super.input_profiles import select_task_input
from alpamayo2_super.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo2_super.models.alpamayo2_super import Alpamayo2Super
from alpamayo2_super.visualization import plot_inference_result

In [ ]:
cwd = Path.cwd()
if (cwd / "examples").exists():
    project_root = cwd
elif (cwd.parent / "examples").exists():
    project_root = cwd.parent
else:
    project_root = cwd

MODEL_ID = os.environ.get("ALPAMAYO2_SUPER_MODEL_ID", PUBLIC_MODEL_ID)
MANIFEST = resolve_project_path(
    os.environ.get("ALPAMAYO2_SUPER_VALIDATION_MANIFEST", "examples/validation_samples.json"),
    project_root,
)
SAMPLE_INDEX = int(os.environ.get("ALPAMAYO2_SUPER_SAMPLE_INDEX", "0"))
DIFFUSION_STEPS = int(os.environ.get("ALPAMAYO2_SUPER_DIFFUSION_STEPS", "10"))
NUM_TRAJ_SAMPLES = int(os.environ.get("ALPAMAYO2_SUPER_NUM_TRAJ_SAMPLES", "1"))
SEED = int(os.environ.get("ALPAMAYO2_SUPER_SEED", "42"))
OUTPUT_DIR = resolve_project_path(
    os.environ.get("ALPAMAYO2_SUPER_OUTPUT_DIR", "outputs"), project_root
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sample = json.loads(MANIFEST.read_text(encoding="utf-8"))["samples"][SAMPLE_INDEX]
clip_id = os.environ.get("ALPAMAYO2_SUPER_CLIP_ID", sample["clip_id"])
t0_us = int(os.environ.get("ALPAMAYO2_SUPER_T0_US", str(sample["t0_us"])))
validate_model_id(MODEL_ID)
if not torch.cuda.is_available():
    raise RuntimeError("Alpamayo 2 Super expert inference requires a CUDA GPU.")

print("model:", MODEL_ID)
print("sample:", SAMPLE_INDEX, clip_id, t0_us)

In [ ]:
source_data = load_physical_aiavdataset(
    clip_id,
    t0_us=t0_us,
)
data = select_task_input(source_data, "trajectory")
print("camera_indices:", data["camera_indices"].tolist())
print("camera_projection_available:", "camera_calibrations" in data)

In [ ]:
model = Alpamayo2Super.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda:0")
model_inputs = helper.prepare_model_inputs(data, model.config, model.tokenizer)
model_inputs = helper.to_device(model_inputs, "cuda")

In [ ]:
torch.cuda.manual_seed_all(SEED)
with torch.autocast("cuda", dtype=torch.bfloat16):
    pred_xyz, pred_rot, logprob, extra = model.sample_trajectories_from_data(
        data=model_inputs,
        top_p=0.98,
        temperature=0.6,
        num_traj_samples=NUM_TRAJ_SAMPLES,
        diffusion_kwargs={"inference_step": DIFFUSION_STEPS},
        return_extra=True,
    )

gt_xy = data["ego_future_xyz"].cpu()[0, 0, :, :2].numpy()
pred_xy = pred_xyz.cpu().numpy()[0, 0, :, :, :2]
distances = np.linalg.norm(pred_xy - gt_xy[None, :, :], axis=-1)
cot_values = np.asarray(extra.get("cot", []), dtype=object)
cots = [str(value) for value in cot_values.reshape(-1)]
print("CoT (per trajectory):", cots)
print("minADE:", distances.mean(axis=-1).min())
print("minFDE:", distances[:, -1].min())

In [ ]:
artifact_stem = f"sample{SAMPLE_INDEX}_{clip_id}_{t0_us}"
png_path = OUTPUT_DIR / f"{artifact_stem}.png"
json_path = OUTPUT_DIR / f"{artifact_stem}.json"
fig, metadata = plot_inference_result(
    data=data,
    pred_xyz=pred_xyz,
    extra=extra,
    output_path=png_path,
    json_path=json_path,
    model_id=MODEL_ID,
    seed=SEED,
)
display(fig)
plt.close(fig)
print("saved_png:", png_path)
print("saved_json:", json_path)
print("projection_available:", metadata["projection_available"])